# ViMed-RAG — Tuần 2: embed + index (Google Colab)

> ⚠️ **NOTEBOOK NÀY CHƯA TỪNG ĐƯỢC CHẠY.** Bản thật sự đã dựng 2 collection là
> `kaggle_build_index.ipynb` (Kaggle T4). File này viết ra lúc còn tưởng Kaggle
> không cứu được, giữ làm **dự phòng khi Kaggle hết quota GPU (30h/tuần)**.
> Dùng nó thì soi lại cell 4 (Colab Secrets) và cell 5 (upload corpus) trước —
> chưa có lần thực thi nào chứng minh chúng đúng.

Bản Colab của `kaggle_build_index.ipynb`.

**Về hai lỗi Kaggle hay gặp** ("không chọn được Accelerator" và "`git clone` fail"):
đó là **một** nguyên nhân duy nhất — tài khoản chưa xác minh số điện thoại, Kaggle
khoá GPU *và* Internet cùng lúc. Xác minh ở Settings → Phone Verification là hết cả
hai; **không cần đổi sang Colab**. Colab chỉ hữu ích khi Kaggle hết quota.

Notebook này **mỏng có chủ đích**: logic nằm trong repo (`src/`, `scripts/`), ở đây chỉ
là dây nối. Sửa logic thì sửa trong repo rồi commit — code trong notebook không có test.

## Khác Kaggle ở đúng 2 chỗ

| | Kaggle | Colab |
|---|---|---|
| Secret | `kaggle_secrets.UserSecretsClient` | `google.colab.userdata` (icon 🔑 thanh trái) |
| Corpus | Dataset gắn vào `/kaggle/input` | upload thẳng, hoặc mount Drive |

## Chuẩn bị

1. **Runtime → Change runtime type → T4 GPU** (bắt buộc, cell 2 sẽ chặn nếu quên).
2. **Secrets (🔑 thanh trái)**: thêm `QDRANT_URL`, `QDRANT_API_KEY`, bật *Notebook access*.
   Chưa làm cũng chạy được — cell 4 sẽ hỏi bằng ô nhập ẩn, không lưu vào output.
3. Có sẵn `data/processed/corpus.jsonl` (12,8 MB) trên máy để upload ở cell 5.

## Kiểm chéo

| size | chunk (DEC-020) | collection |
|---|---|---|
| 512 | 4.972 | `vimed_rag_512` |
| 256 | 10.246 | `vimed_rag_256` |

Session Colab bị ngắt giữa chừng **không sao**: point ID tất định (UUID5 của
`doc_id:chunk_idx`) nên chạy lại là upsert đè, không nhân đôi.

### 1. Kiểm GPU — dừng sớm nếu runtime sai

Không có GPU thì đừng chạy tiếp: đã đo trên CPU laptop là **0,22 chunk/s → ~12 giờ**
cho cả hai size. CPU của Colab cũng không khá hơn bao nhiêu.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "DỪNG LẠI: chưa bật GPU. Runtime -> Change runtime type -> "
        "Hardware accelerator = T4 GPU, rồi chạy lại từ cell này."
    )

print("GPU  :", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)
!nvidia-smi --query-gpu=memory.total,memory.free --format=csv

### 2. Cài dependency + lấy code

`FlagEmbedding` có thể kéo theo torch bản khác. Nếu Colab báo cần restart:
**Runtime → Restart session**, rồi chạy lại từ cell 1.

In [ ]:
!pip install -q "FlagEmbedding>=1.2,<2.0" "qdrant-client>=1.9"

In [ ]:
import os, shutil

REPO_DIR = "/content/vimed-rag"
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)  # luôn lấy bản mới nhất, tránh chạy nhầm code cũ

!git clone --depth 1 https://github.com/dateddy/vimed-rag.git {REPO_DIR}
os.chdir(REPO_DIR)
!git log --oneline -1

### 3. Secret → biến môi trường

Ưu tiên Colab Secrets. Chưa cấu hình thì rơi xuống `getpass` — gõ tay, **không hiện
ra màn hình và không lưu vào output notebook**. Tuyệt đối đừng gán thẳng key vào cell.

In [ ]:
from getpass import getpass

def load_secret(name: str) -> str:
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            print(f"{name}: lấy từ Colab Secrets")
            return val
    except Exception:
        pass
    return getpass(f"{name} (gõ/dán, không hiện ra): ")

os.environ["QDRANT_URL"] = load_secret("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = load_secret("QDRANT_API_KEY")

# In có che — đủ để biết đúng cluster, không lộ key.
print("URL :", os.environ["QDRANT_URL"][:45] + "...")
print("KEY :", "đã nạp" if os.environ["QDRANT_API_KEY"] else "RỖNG")

!python scripts/smoke_qdrant.py

### 4. Đưa corpus lên

`corpus.jsonl` bị gitignore nên **không có sẵn** sau khi clone. Upload từ máy
(12,8 MB, ~10 giây).

*Vì sao không chạy lại ingestion ở đây:* DEC-020 (4.972 chunk, 3 tiêu chí PASS) đo trên
**đúng file này**. Sinh lại corpus = số của DEC-020 hết hiệu lực mà không ai biết.

Chạy nhiều lần trong ngày thì thay bằng Drive cho khỏi upload lại:
```python
from google.colab import drive; drive.mount('/content/drive')
CORPUS = '/content/drive/MyDrive/vimed-rag/corpus.jsonl'
```

In [ ]:
CORPUS = "/content/corpus.jsonl"

if os.path.exists(CORPUS):
    print("Đã có sẵn từ lần chạy trước, không cần upload lại.")
else:
    from google.colab import files

    print("Chọn data/processed/corpus.jsonl trên máy bạn...")
    uploaded = files.upload()          # files.upload() ghi vào thư mục hiện tại
    name = next(iter(uploaded))        # = /content/vimed-rag sau khi os.chdir
    shutil.move(os.path.join(os.getcwd(), name), CORPUS)

size_mb = os.path.getsize(CORPUS) / 1024**2
print(f"OK: {CORPUS} ({size_mb:.1f} MB)")
if not 12.0 < size_mb < 14.0:
    print("!! Kích thước lệch so với 12,8 MB đã biết — kiểm lại đúng file chưa.")

### 5. Dry-run — kiểm đường ống trước khi đụng GPU

Phải ra **4.972 chunk** và **chunk 2 khoa > 0**. Sai một trong hai thì dừng, đừng chạy tiếp.

In [ ]:
!python scripts/build_index.py --size 512 --corpus {CORPUS} --dry-run

### 6. Index chunk 512 (~4.972 point)

Lần chạy đầu tải bge-m3 (~2,3 GB) nên chậm hơn. Theo dõi `chunk/s` + ETA.

In [ ]:
!python scripts/build_index.py --size 512 --corpus {CORPUS}

In [ ]:
!python scripts/verify_index.py --size 512

### 7. Index chunk 256 (~10.246 point) — ablation DEC-004

Chỉ chạy khi cell verify ở trên đã **PASS**.

In [ ]:
!python scripts/build_index.py --size 256 --corpus {CORPUS}

In [ ]:
!python scripts/verify_index.py --size 256

### 8. Ghi lại số thực để đóng Tuần 2

Chép output cell dưới vào `brain/state/STATUS.md`. Ngân sách chốt ở DEC-016 là **~8%
free tier 1GB cho cả 2 collection** — lệch xa thì phải ghi một dòng DECISIONS mới,
đừng để lệch âm thầm.

In [ ]:
from qdrant_client import QdrantClient

client = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"], timeout=60)
for c in client.get_collections().collections:
    print(f"{c.name:20s} {client.count(c.name, exact=True).count:7d} point")